# 03 - Provision PostgreSQL + pgvector & Chunk Labels

**Learning objective:** Provision a managed database and populate it with section-typed drug label chunks.

We will:
1. Live-provision Scaleway Managed PostgreSQL via `tofu apply` (~5-8 min)
2. Connect with `psycopg` and enable the `vector` extension
3. Create the chunks table with metadata columns
4. Download labels from S3, chunk them, and bulk-insert
5. Verify chunk counts per section type

In [ ]:
import os
import subprocess
import json
from dotenv import load_dotenv

load_dotenv()

project_suffix = os.environ["PROJECT_SUFFIX"]
iac_dir = "../iac_snippets/postgres"

# Write tfvars
tfvars = f"""access_key      = "{os.environ["SCW_ACCESS_KEY"]}"
secret_key      = "{os.environ["SCW_SECRET_KEY"]}"
organization_id = "{os.environ["SCW_DEFAULT_ORGANIZATION_ID"]}"
project_id      = "{os.environ["SCW_DEFAULT_PROJECT_ID"]}"
project_suffix      = "{project_suffix}"
"""
with open(f"{iac_dir}/terraform.tfvars", "w") as f:
    f.write(tfvars)

print("Written terraform.tfvars")

In [ ]:
# Initialize and apply (~5-8 minutes)
print("Provisioning PostgreSQL... this takes ~5-8 minutes.")
subprocess.run(["tofu", "init"], cwd=iac_dir, check=True)
result = subprocess.run(
    ["tofu", "apply", "-auto-approve"],
    cwd=iac_dir,
    capture_output=True,
    text=True,
)
print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
    raise RuntimeError(f"tofu apply failed (rc={result.returncode})")

In [ ]:
# Get connection details from tofu output
def get_output(name):
    r = subprocess.run(["tofu", "output", "-raw", name], cwd=iac_dir, capture_output=True, text=True)
    return r.stdout.strip()


db_host = get_output("host")
db_port = get_output("port")
db_name = get_output("database")
db_user = get_output("user")
db_password = get_output("password")

print(f"Database: {db_host}:{db_port}/{db_name}")

###  Connect to PostgreSQL & Enable pgvector

This cell connects to **Scaleway Managed PostgreSQL** and activates the `pgvector` extension -
which adds vector similarity search capabilities to a standard relational database.

Once enabled, PostgreSQL can store **embedding vectors** alongside regular data
and run fast nearest-neighbour searches - no separate vector database needed.


In [ ]:
# Connect and enable pgvector
import psycopg

conn = psycopg.connect(
    host=db_host,
    port=int(db_port),
    dbname=db_name,
    user=db_user,
    password=db_password,
    sslmode="require",
)

with conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector")
    conn.commit()
    cur.execute("SELECT * FROM pg_extension WHERE extname='vector'")
    row = cur.fetchone()
    print(f"pgvector installed: {row is not None}")

### Create the Chunks Table

This cell creates the `chunks` table in PostgreSQL - the core storage structure for our RAG pipeline. The code imports `create_table`  from the **`src/rag.py`**


###  Table Schema: `chunks`

```sql
chunks (
    id                 SERIAL PRIMARY KEY,
    drug_name          TEXT NOT NULL,
    generic_name       TEXT NOT NULL,
    brand_name         TEXT NOT NULL,
    section_type       TEXT NOT NULL,
    set_id             TEXT NOT NULL,
    application_number TEXT,
    manufacturer_name  TEXT,
    source_url         TEXT,
    text               TEXT NOT NULL,
    embedding          vector(768)
);
```

In [ ]:
# Create the chunks table
import sys

sys.path.insert(0, "..")

from src.rag import create_table, insert_chunks

create_table(conn)
print("Chunks table created.")

In [ ]:
# Download labels from S3 and chunk them
import boto3

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://s3.{os.environ['SCW_DEFAULT_REGION']}.scw.cloud",
    aws_access_key_id=os.environ["SCW_ACCESS_KEY"],
    aws_secret_access_key=os.environ["SCW_SECRET_KEY"],
    region_name=os.environ["SCW_DEFAULT_REGION"],
)

bucket_result = subprocess.run(
    ["tofu", "output", "-raw", "bucket_name"],
    cwd="../iac_snippets/object_storage",
    capture_output=True,
    text=True,
    check=True,
)
bucket_name = bucket_result.stdout.strip()
obj = s3.get_object(Bucket=bucket_name, Key="openfda_labels.json")
labels = json.loads(obj["Body"].read())
print(f"Downloaded {len(labels)} labels from S3.")

###  Chunk All Labels & Insert into PostgreSQL

This cell processes all 200 FDA labels, splits them into text chunks, and stores them in the database. 

1 chunk = 1 FDA label section (e.g. the full drug_interactions, the full boxed_warning)

Finally insert chunks into table chunks.

In [ ]:
# Chunk all labels and insert (without embeddings for now)
from src.chunker import chunk_label

all_chunks = []
for label in labels:
    chunks = chunk_label(label)
    for chunk in chunks:
        chunk["embedding"] = None  # Embeddings added in notebook 04
    all_chunks.extend(chunks)

print(f"Total chunks: {len(all_chunks)}")

# Bulk insert
insert_chunks(conn, all_chunks)
print("Chunks inserted into pgvector.")

### ✅ You should now have

- ☑️ A running Managed PostgreSQL instance with pgvector enabled
- ☑️ A `chunks` table with full metadata columns
- ☑️ Hundreds of section-typed chunks from the openFDA dataset
- ☑️ Verified chunk counts for `drug_interactions`, `pregnancy`, `boxed_warning` and more

> 🚀 **Next:** `04_embeddings_and_search.ipynb`